- notebook to compare against baseline models 



In [ ]:
import polars as pl
import numpy as np

df = pl.read_parquet("cq_final_features.parquet")

In [ ]:
FEATURES = [
    "hour",
    "weekday",
    "speed_mean",
    "distance_travelled",
    "gps_gap_min",
    "workload_capped",
    "batch_rank_capped",
    "late_batch",
    "high_load",
    "overloaded",
]

TARGET = "eta_mins"

X = df.select(FEATURES).to_numpy()
y = df[TARGET].to_numpy()

In [ ]:
# not random split 

split = int(len(X)*0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("RF MAE:",
      mean_absolute_error(y_test, pred_rf))

In [ ]:
!pip install lightgbm

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8
)

lgbm.fit(X_train, y_train)

pred_lgb = lgbm.predict(X_test)

print("LightGBM MAE:",
      mean_absolute_error(y_test, pred_lgb))

In [ ]:
import pandas as pd

imp = pd.Series(
    lgbm.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print(imp)

Contextual bandits assume reward = nonlinear(state) Random forest validate this assumption 
